## Imports & raw download

In [31]:
!pip install -q lxml          # 250 kB, C-accelerated, fastest

In [32]:
import pandas as pd
import numpy as np
from urllib.request import urlopen

# --- raw HTML -----------------------------------------------------------------
URL = "https://www.basketball-reference.com/leagues/NBA_2024_totals.html"  # season totals
tables = pd.read_html(urlopen(URL), header=0)        # returns list of all <table>
totals   = tables[0]                                    # first table = "Player Totals"
per_game = tables[1]                               # second table = "Per Game statistics"


print(f"Raw totals shape: {totals.shape}")
totals

Raw totals shape: (736, 32)


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Awards
0,1.0,Luka Dončić,24.0,DAL,PG,70.0,70.0,2624.0,804.0,1652.0,0.487,284.0,744.0,0.382,520.0,908.0,0.573,0.573,478.0,608.0,0.786,59.0,588.0,647.0,686.0,99.0,38.0,282.0,149.0,2370.0,21.0,"MVP-3,CPOY-6,AS,NBA1"
1,2.0,Shai Gilgeous-Alexander,25.0,OKC,PG,75.0,75.0,2553.0,796.0,1487.0,0.535,95.0,269.0,0.353,701.0,1218.0,0.576,0.567,567.0,649.0,0.874,65.0,350.0,415.0,465.0,150.0,67.0,162.0,184.0,2254.0,0.0,"MVP-2,DPOY-7,CPOY-3,AS,NBA1"
2,3.0,Giannis Antetokounmpo,29.0,MIL,PF,73.0,73.0,2567.0,837.0,1369.0,0.611,34.0,124.0,0.274,803.0,1245.0,0.645,0.624,514.0,782.0,0.657,196.0,645.0,841.0,476.0,87.0,79.0,250.0,210.0,2222.0,10.0,"MVP-4,DPOY-9,CPOY-12,AS,NBA1"
3,4.0,Jalen Brunson,27.0,NYK,PG,77.0,77.0,2726.0,790.0,1648.0,0.479,211.0,526.0,0.401,579.0,1122.0,0.516,0.543,421.0,497.0,0.847,43.0,235.0,278.0,519.0,70.0,13.0,186.0,144.0,2212.0,0.0,"MVP-5,CPOY-5,AS,NBA2"
4,5.0,Nikola Jokić,28.0,DEN,C,79.0,79.0,2737.0,822.0,1411.0,0.583,83.0,231.0,0.359,739.0,1180.0,0.626,0.612,358.0,438.0,0.817,223.0,753.0,976.0,708.0,108.0,68.0,237.0,194.0,2085.0,25.0,"MVP-1,CPOY-4,AS,NBA1"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,569.0,Ron Harper Jr.,23.0,TOR,PF,1.0,0.0,4.0,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,NaN
732,570.0,Justin Jackson,28.0,MIN,SF,2.0,0.0,1.0,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
733,571.0,Dmytro Skapintsev,25.0,NYK,C,2.0,0.0,2.0,0.0,1.0,0.000,0.0,0.0,NaN,0.0,1.0,0.000,0.000,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
734,572.0,Javonte Smart,24.0,PHI,PG,1.0,0.0,1.0,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


## Data cleaning, filtering:

In [33]:
# ------------------------------------------------------------------ 1. FIRST-PASS CLEAN ---------------------------
totals = totals[~totals["Player"].str.contains("League Average", na=False)]
totals = totals.dropna(subset=["Player"]).reset_index(drop=True)

num_cols = totals.columns.difference(["Player", "Pos", "Tm"])
totals[num_cols] = totals[num_cols].apply(pd.to_numeric, errors="coerce")

# ------------------------------------------------------------------ 2. KEEP ONE ROW PER PLAYER --------------------
multi = totals.loc[totals.duplicated("Player", keep=False), "Player"].unique()
is_tot = totals["Team"] == "TOT"
totals = totals.loc[is_tot | (~totals["Player"].isin(multi))].reset_index(drop=True)

# ------------------------------------------------------------------ 3. USAGE FILTERS ------------------------------
totals = totals[(totals["G"] >= 20) & (totals["3PA"] >= 30)].reset_index(drop=True)

# ------------------------------------------------------------------ 4. FINAL TIDY SET -----------------------------
keep = ["Player", "Pos", "Age", "G", "3P", "3PA", "PTS", "TRB", "AST"]
nba24 = totals[keep].copy()
nba24["threes_made"]   = nba24["3P"].astype(int)
nba24["threes_trials"] = nba24["3PA"].astype(int)

nba24.to_csv("nba24_totals_clean.csv", index=False)
print("🟢  Saved nba24_totals_clean.csv  |  rows =", len(nba24))

nba24

🟢  Saved nba24_totals_clean.csv  |  rows = 314


,Player,Pos,Age,G,3P,3PA,PTS,TRB,AST,threes_made,threes_trials
0,Luka Dončić,PG,24.0,70.0,284.0,744.0,2370.0,647.0,686.0,284,744
1,Shai Gilgeous-Alexander,PG,25.0,75.0,95.0,269.0,2254.0,415.0,465.0,95,269
2,Giannis Antetokounmpo,PF,29.0,73.0,34.0,124.0,2222.0,841.0,476.0,34,124
3,Jalen Brunson,PG,27.0,77.0,211.0,526.0,2212.0,278.0,519.0,211,526
4,Nikola Jokić,C,28.0,79.0,83.0,231.0,2085.0,976.0,708.0,83,231
...,...,...,...,...,...,...,...,...,...,...,...
309,Onuralp Bitim,SG,24.0,23.0,12.0,44.0,80.0,32.0,13.0,12,44
310,Markieff Morris,PF,34.0,26.0,15.0,42.0,66.0,39.0,16.0,15,42
311,Cory Joseph,PG,32.0,26.0,13.0,42.0,63.0,30.0,42.0,13,42
312,Justin Minaya,SF,24.0,34.0,12.0,49.0,61.0,56.0,21.0,12,49
